# Udemy Purchase Scraper

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from undetected_chromedriver import By
from decimal import Decimal
from dotenv import load_dotenv, find_dotenv
import undetected_chromedriver as uc
import os
import datetime
import time
import json
import re
import requests

import pickle
from copy import deepcopy

In [ ]:
# Load environment variables
load_dotenv()

# Options for selenium
options = uc.ChromeOptions()
chrome_version = os.environ.get("CHROME_VERSION", None)
if chrome_version is not None:
    if chrome_version.lower == "none":
        chrome_version = None
    else:
        try:
            chrome_version = int(chrome_version)
        except:
            chrome_version = None
if chrome_version is not None:
    driver = uc.Chrome(use_subprocess=False, options=options, version_main=chrome_version)
else:
    driver = uc.Chrome(use_subprocess=False, options=options)

Manually log into Udemy now because trying to log in automatically with 2FA and everything will be a pain in the ass. Save the cookies to skip this step later (and prevent being locked out if you log in too many times).

## Utility

In [ ]:
def saveCookies(driver: uc.Chrome, f_path: str, force_cookies: bool = True) -> None:
    '''
    Saves the cookies from your udemy in

    :param driver: The web driver
    :type driver: Chrome

    :param f_path: The file name (or full file path)
    :type f_path: str

    :param force_cookies: If we force the cookies even if they already exist, defaults to [True]
    :type force_cookies: bool

    :return: Nothing
    :rtype: None
    '''
    # Checking if the cookies file already exists
    if(force_cookies or not os.path.exists(f_path)):
        # Wait for an input for the user to log in
        input("Continue after you've logged into Udemy!")
        # To save cookies
        saved_cookies = driver.get_cookies()
        with open(f_path, "w") as file:
            json.dump(saved_cookies, file)
    return None

In [ ]:
def loadCookies(driver: uc.Chrome, f_path: str):
    '''
    Loads your Udemy cookies in

    :param driver: The web driver
    :type driver: Chrome

    :param f_path: The file name (or full file path)
    :type f_path: str

    :return: Nothing
    :rtype: None
    '''
    # To load in cookies
    with open(f_path, "r") as file:
        saved_cookies = json.load(file)
    for x in saved_cookies:
        driver.add_cookie(x)
    # Wait for one second just to be sure (probably not needed)
    time.sleep(1)
    return None

In [ ]:
@dataclass
class UdemyCoursePurchase:
    '''
    A class to store Udemy courses
    '''
    course_name: str
    course_link: str
    course_cost: Decimal

In [ ]:
@dataclass
class UdemyCoursePurchaseDetails:
    course_purchase: str
    course_date: datetime.date
    course_cost: Decimal

In [ ]:
@dataclass
class UdemyCourseOwn:
    '''
    A class to store Udemy courses
    '''
    course_name: str
    course_link: str
    course_author: str
    course_page: str|None = None
    purchase: UdemyCoursePurchaseDetails|None = None

In [ ]:
@dataclass
class UdemyPurchase:
    '''
    A class to store udemy purchases
    '''
    purchase_number: str
    purchase_date: datetime.date
    purchase_cost: Decimal
    courses: any|None = None

## Purchases

In [ ]:
def getPurchasePage(driver: any) -> any:
    '''
    A function to get the purchases from the 'purchase-history' page
    '''
    purchase_table = driver.find_element(By.XPATH, "//table[@class='paginated-table_purchase-history-table__ncCll']")
    purchases = purchase_table.find_elements(By.XPATH, ".//tr[@class='paginated-table_purchase-history-table--main-row__5hHCb']")
    return purchases

In [ ]:
def getPurchaseInformation(purchase: any) -> any:
    '''
    
    '''
    purchase_number = purchase.find_element(By.XPATH, ".//span[@class='ud-btn-label' and contains(., 'Receipt')]")
    purchase_number = purchase_number.find_element(By.XPATH, "..")
    purchase_number = purchase_number.get_attribute("href")
    purchase_number = purchase_number.split("/")[-2]

    purchase_cost = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-history-table--data-cell__LO_7L']/div[@class='app_positive-text___Wb07']").text
    purchase_cost = Decimal(purchase_cost.replace("$","").strip())

    purchase_date = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-history-table--data-cell__LO_7L' and not(descendant::*)]").text
    purchase_date = datetime.datetime.strptime(purchase_date, "%b %d, %Y").date()

    cur_purchase = UdemyPurchase(purchase_number=purchase_number,
                                 purchase_cost= purchase_cost,
                                 purchase_date=purchase_date)
    return(cur_purchase)

In [ ]:
class SingleCourseError(Exception):
    '''Raised when a purchase only has a single course'''

def checkMultipleCourses(purchase: any) -> any:
    '''
    A function to check if a purchase has multiple courses
    '''
    try:
        link_section = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-details--text-container__EvFRz']")
        show_more_button = link_section.find_element(By.XPATH, ".//button[@class='ud-btn ud-btn-medium ud-btn-link ud-btn-text-sm show-more_focusable-label__dmEhW']")
        return(show_more_button)
    except:
        raise SingleCourseError("Purchase only has a single course")

In [ ]:
def getPurchaseCourseSingle(purchase: any) -> any:
    '''
    A function to get the courses from a purchase with a single course
    '''
    course_name_element = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-details--text__JKVm6 paginated-table_purchase-details--content-title-text__DxDuf ud-text-md']")
    course_name_element = course_name_element.find_element(By.XPATH, ".//a")

    course_name = course_name_element.get_attribute("innerHTML")
    course_link = course_name_element.get_attribute("href")
    
    course_price = purchase.find_element(By.XPATH, ".//div[@class='app_positive-text___Wb07']").text
    course_price = Decimal(course_price.replace("$","").strip())
    
    course = UdemyCoursePurchase(course_name=course_name,
                                 course_link=course_link,
                                 course_cost=course_price)
    return(course)


In [ ]:
def getPurchaseCourseMultiple(driver: any) -> any:
    '''
    A function to get the courses from a purchase with multiple courses
    '''
    courses = driver.find_elements(By.XPATH, ".//tr[@class='paginated-table_purchase-history-table--sub-row__2g87b']")
    courses = [getPurchaseCourseSingle(purchase=course) for course in courses]
    return(courses)
    

In [ ]:
def getPurchaseCourses(purchase: any, driver: any) -> any:
    '''
    A function to get the courses in a purchase
    '''
    try:
        expand_purchase = checkMultipleCourses(purchase)
        expand_purchase.click()
        time.sleep(int(os.environ.get("SLEEP_TIME", 3))/2)
        courses = getPurchaseCourseMultiple(driver=driver)
        expand_purchase.click()
        time.sleep(int(os.environ.get("SLEEP_TIME", 3))/2)
    except SingleCourseError:
        courses = [getPurchaseCourseSingle(purchase)]
    return(courses)
        

In [ ]:
class RetryError(Exception):
    '''Raised when there are no more retries'''

def getPurchasePageResults(driver: any) -> any:
    """
    A function to get the results from a page of purchases
    """
    retry = 0
    while retry < int(os.environ.get("RETRY_LIMIT", 3)):
        try:
            purchase_page = getPurchasePage(driver=driver)
            purchases = [None] * len(purchase_page)
            for ind, purchase in enumerate(purchase_page):
                cur_purchase: UdemyPurchase = getPurchaseInformation(purchase=purchase)
                cur_purchase.courses = getPurchaseCourses(purchase=purchase, driver=driver)
                purchases[ind] = cur_purchase
            return(purchases)
        except:
            retry += 1
            time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    raise RetryError("Ran out of retries")

In [ ]:
class LastPageError(Exception):
    '''Raised when there are no more purchase pages'''

def checkMorePurchases(driver: any) -> None:
    '''
    A function to check if a purchase has multiple courses
    '''
    try:
        next_page = driver.find_element(By.XPATH, ".//a[@class='ud-btn ud-btn-medium ud-btn-secondary ud-btn-round ud-btn-text-sm ud-btn-icon ud-btn-icon-medium ud-btn-icon-round pagination_next__aBqfT' " \
                                                    "and @aria-label='next page' " \
                                                    "and @aria-disabled='false']")
        next_page.click()
        time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    except:
        raise LastPageError("No more purchase pages")

In [ ]:
def getPurchasePages(driver: any) -> any:
    """
    A function to get the purchase results from all pages
    """
    driver.get("https://www.udemy.com/dashboard/purchase-history/")
    time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    purchases = []
    while True:
        page_results = getPurchasePageResults(driver)
        #purchases.append(page_results)
        purchases.extend(page_results)
        try:
            checkMorePurchases(driver=driver)
        except LastPageError:
            break
    return(purchases)

## Loading

In [ ]:
driver.get("https://www.udemy.com/")
time.sleep(int(os.environ.get("SLEEP_TIME", 3)))

In [ ]:
cookies_path = os.environ.get("COOKIES_PATH", "cookies.json")
force_new_cookies = os.environ.get("FORCE_NEW_COOKIES", False)
if isinstance(force_new_cookies, str):
    if force_new_cookies.lower() == "false":
        force_new_cookies = False
    elif force_new_cookies.lower() == "true":
        force_new_cookies = True
    else:
        force_new_cookies = False
saveCookies(driver=driver, f_path=cookies_path, force_cookies=force_new_cookies)
loadCookies(driver=driver, f_path=cookies_path)
time.sleep(int(os.environ.get("SLEEP_TIME", 3)))

## Get Purchases

In [ ]:
purchases = getPurchasePages(driver=driver)

In [ ]:
backup_purc = deepcopy(purchases)

In [ ]:
unique_courses = {}
for purchase in purchases:
    purchase.courses = [course for course in purchase.courses if course.course_link not in unique_courses]
    for course in purchase.courses:
        if course.course_link not in unique_courses:
            unique_courses[course.course_link] = None
del unique_courses

## Ownership

In [ ]:
def getOwnedInformation(course: any) -> any:
    '''
    
    '''
    course_info = course.find_element(By.XPATH,".//h3[@data-purpose='course-title-url']//a")
    course_title = course_info.text
    course_link = course_info.get_attribute("href")
    instructor = course.find_element(By.XPATH, ".//div[@class='course-card-instructors-module--instructor-list--cJTfw']").text
    instructor = re.split(r"[,\|•-]", instructor)[0].strip()
    cur_purchase = UdemyCourseOwn(course_name=course_title,
                                  course_link=course_link,
                                  course_author=instructor)
    return(cur_purchase)

In [ ]:
def getOwnedPage(driver: any) -> any:
    '''
    
    '''
    courses_grid = driver.find_element(By.XPATH, "//div[@class='my-courses__course-card-grid']")
    courses = courses_grid.find_elements(By.XPATH, "./div")
    courses = [getOwnedInformation(course=course) for course in courses]
    return(courses)

In [ ]:
def checkMoreOwned(driver: any) -> None:
    '''
    
    '''
    try:
        next_page = driver.find_element(By.XPATH, ".//a[@class='ud-btn ud-btn-medium ud-btn-secondary ud-btn-round ud-btn-text-sm ud-btn-icon ud-btn-icon-medium ud-btn-icon-round pagination-module--next--QX6nm'" \
                                        " and @aria-label='next page' " \
                                        "and @aria-disabled='false']")
        next_page.click()
        time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    except:
        raise LastPageError("No more owned pages")

In [ ]:
def getOwnedResults(driver: any) -> any:
    """
    
    """
    retry = 0
    while retry < int(os.environ.get("RETRY_LIMIT", 3)):
        try:
            page_results = getOwnedPage(driver=driver)
            return(page_results)
        except:
            retry += 1
            time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    raise RetryError("Ran out of retries")

In [ ]:
def getOwnedPages(driver: any) -> any:
    """
    
    """
    driver.get("https://www.udemy.com/home/my-courses/learning/?sort=-enroll_time")
    time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    courses = []
    while True:
        page_results = getOwnedResults(driver=driver)
        courses.extend(page_results)
        try:
            checkMoreOwned(driver=driver)
        except LastPageError:
            break
    return(courses)

## Get Ownership

In [ ]:
# If I had any archived courses I would also want to check those. Should be the same functions and then just extend the lists.
owned_courses = getOwnedPages(driver)

In [ ]:
unique_courses = {}
# new_courses = []
# for course in owned_courses:
#     if course.course_link not in unique_courses:
#         unique_courses[course.course_link] = None
#         new_courses.append(course)
ind = 0
while ind < len(owned_courses):
    c_link = owned_courses[ind].course_link
    if(c_link not in unique_courses):
        unique_courses[c_link] = None
        ind = ind + 1
    else:
        del owned_courses[ind]
del unique_courses

In [ ]:
backup_owned = deepcopy(owned_courses)

## Get Owned Real URL

In [ ]:
def getOwnedCurrentURL(driver: any, course_link: str):
    """
    x
    """
    driver.get(course_link)
    #time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    return(driver.current_url.split("/learn")[0])

In [ ]:
def requestsCookiesFromSelenium(cookies_file: str) -> any: #selenium_cookies: dict) -> any:
    '''
    x
    '''
    with open(cookies_file, "r") as file:
        saved_cookies = json.load(file)
    jar = requests.cookies.RequestsCookieJar()
    for cookie in saved_cookies:
        rest = {}
        for key, val in cookie.items():
            if key not in ["name", "value", "domain"]:
                rest[key] = val
        jar.set_cookie(
                requests.cookies.create_cookie(
                    name = cookie.get("name"),
                    value = cookie.get("value"),
                    domain = cookie.get("domain"),
                    #expiry = cookie.get("expiry"),
                    #httpOnly = cookie.get("httpOnly"),
                    #path = cookie.get("path"),
                    #sameSite = cookie.get("sameSite"),
                    #secure = cookie.get("secure")
                    rest = rest  
                )
            )
    return(jar)

In [ ]:
requests_cookies = requestsCookiesFromSelenium(cookies_file=cookies_path)

In [ ]:
def getOwnedCurrentURLRequest(session: any, course_link: str):
    """
    x
    """
    # #response = session.get(course_link, cookies=jar).url
    # #response = session.head(course_link, allow_redirects=True).url
    # response = session.head(course_link, allow_redirects=True)
    # print(response.status_code)
    # #time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    # #return(response.split("/learn")[0])
    # return(response.url.split("/learn")[0])
    retry = 0
    while retry < int(os.environ.get("RETRY_LIMIT", 3)):
        try:
            response = session.head(course_link, allow_redirects=True)
            response.raise_for_status()
            return(response.url.split("/learn/")[0])
        except:
            retry += 1
            time.sleep(int(os.environ.get("SLEEP_TIME", 3)))
    raise RetryError("Ran out of retries")

## Get URLS

In [ ]:
with requests.Session() as session:
    #session.headers.update(requests_cookies)
    session.cookies = requests_cookies #requests.utils.dict_from_cookiejar(jar)
    for ind, course in enumerate(owned_courses):
        owned_courses[ind].course_page = f"{getOwnedCurrentURLRequest(session=session, course_link=course.course_link)}/"

## Temporary Data Serialization

In [ ]:
data_save = {"purchases": purchases, "owned_courses": owned_courses}

with open("backup_data.pkl", "wb") as save_file:
    pickle.dump(data_save, save_file)

## Temporary Data Loading

In [ ]:
with open("backup_data.pkl", "rb") as load_file:
    data_load = pickle.load(load_file)

purchases = data_load["purchases"]
owned_courses = data_load["owned_courses"]

## Get Dictionaries

In [ ]:
owned_courses_dict = {}
for course in owned_courses:
    owned_courses_dict[course.course_page] = course

In [ ]:
purchase_courses_dict = {}
for purchase in purchases:
    for course in purchase.courses:
        purchase_courses_dict[course.course_link] = None

In [ ]:
missing_owned_courses = set(owned_courses_dict.keys()).difference(set(purchase_courses_dict.keys()))
free_courses = [owned_courses_dict[course] for course in missing_owned_courses]
del purchase_courses_dict

In [ ]:
original_lengths = [len(purchase.courses) for purchase in purchases]

for purchase in purchases:
    purchase.courses = [course for course in purchase.courses if course.course_link in owned_courses_dict]

new_lengths = [len(purchase.courses) for purchase in purchases]

In [ ]:
for ind, purchase in enumerate(purchases):
    if original_lengths[ind] != new_lengths[ind] and new_lengths[ind] != 0:
        purchases[ind].purchase_cost = sum([course.course_cost for course in purchases[ind].courses])
del original_lengths, new_lengths

purchases = [purchase for purchase in purchases if len(purchase.courses) != 0]

In [ ]:
course_prices = {}
for purchase in purchases:
    for course in purchase.courses:
        course_prices[course.course_link] = UdemyCoursePurchaseDetails(
            course_purchase=purchase.purchase_number, 
            course_date=purchase.purchase_date, 
            course_cost=course.course_cost)

In [ ]:
for crs_ind, course in enumerate(owned_courses):
    if course.course_page in course_prices:
        owned_courses[crs_ind].purchase = course_prices[course.course_page]
del course_prices

In [ ]:
for p_ind, purchase in enumerate(purchases):
    for c_ind, course in enumerate(purchase.courses):
        purchases[p_ind].courses[c_ind] = owned_courses_dict[course.course_link]
del owned_courses_dict

## Printing

In [ ]:
purchases

In [ ]:
owned_courses